# Figure 4 — aligned ChIP-seq, genes, and Micro-C

Six ChIP tracks follow the order listed in the supplied draft, replacing the older larval-brain tracks with the matched 2026 WT/RNAi samples. “RNAi” denotes **Sry-delta RNAi** for both antibodies, based on the accompanying message.

**Review before running:** the reference image has no genomic coordinates or Micro-C sample labels. Panels a–c below are windows inferred from the displayed gene neighborhoods, not exact recovered boundaries or confirmed promoter identities. Panel d is RpS12 with ±20 kb. Embryo and larval-brain Micro-C are selected here; edit them if the draft used different maps. Each panel exports separately as PNG and PDF; this notebook does not assemble the article layout.

Track order: six ChIP profiles → FlyBase r6.59 genes → boundary intervals → two Micro-C maps → coordinate axis. No blue interval bar. WT/RNAi pairs share signal limits per antibody within each panel. Embryo tracks are scaled separately; these files have not been renormalized and cross-run signal equivalence is not assumed.


## Portable paths

Launch Jupyter from the repository root. Input files are expected under `data/` by default. Set `SRY_DATA_DIR`, `SRY_MCOOL_DIR` and (for Fig. 4) `SRY_GTF`, or edit the configuration cell to use other locations. Inputs are not bundled with this notebook. See `README.md` for the directory layout. Source data filenames and analysis parameters are retained. Saved outputs were cleared for publication; no analysis was rerun.


In [ ]:
from pathlib import Path
import os, subprocess, math, shutil, sys
import re
# Run from the repository root; override paths here or with environment variables.
PROJECT_DIR = Path.cwd()
BASE = Path(os.environ.get('SRY_DATA_DIR', PROJECT_DIR / 'data')).expanduser().resolve()
MCOOL_DIR = Path(os.environ.get('SRY_MCOOL_DIR', BASE / 'mcool')).expanduser().resolve()
BOUNDARIES = BASE / '101626' / 'cluster2 embryo CellGen prom prox elements.bed'
OUTPUT = BASE / 'figures' / 'Figure4_pyGenomeTracks'
GTF = Path(os.environ.get('SRY_GTF', BASE / 'annotations' / 'dmel-all-r6.59.gtf')).expanduser().resolve()
candidate = Path(sys.executable).parent / 'pyGenomeTracks'
executable = str(candidate) if candidate.is_file() else shutil.which('pyGenomeTracks')
if executable is None:
    raise RuntimeError('Install pyGenomeTracks and pyBigWig in the notebook environment.')
PYGENOMETRACKS = Path(executable)
PYTHON = Path(sys.executable)

PANELS = {
    'a': dict(label='alphaTub84B–Alh neighborhood', region='chr3R:7082000-7135000'),
    'b': dict(label='bmm–Tdrd3–Helz neighborhood', region='chr3L:14776000-14804000'),
    'c': dict(label='Fkbp59–RpS2 neighborhood', region='chr2L:9886000-9899000'),
    'd': dict(label='RpS12 ±20 kb', region='chr3L:13003352-13044762'),
}
CHIP = [
    dict(id='embryo_sry', title='Embryo WT | Sry-delta', prefix='Galaxy4478-', group='embryo_sry'),
    dict(id='embryo_beaf', title='Embryo WT | BEAF32', prefix='Galaxy4477-', group='embryo_beaf'),
    dict(id='lb_sry_wt', title='LB 2026 WT | Sry-delta', prefix='Galaxy4190-', group='lb_sry'),
    dict(id='lb_sry_rnai', title='LB 2026 Sry-delta RNAi | Sry-delta', prefix='Galaxy4195-', group='lb_sry'),
    dict(id='lb_beaf_wt', title='LB 2026 WT | BEAF32', prefix='Galaxy4189-', group='lb_beaf'),
    dict(id='lb_beaf_rnai', title='LB 2026 Sry-delta RNAi | BEAF32', prefix='Galaxy4194-', group='lb_beaf'),
]
MICROC = [
    dict(id='embryo', title='Embryo Micro-C', path=MCOOL_DIR / 'AllWT_MJP_balanced.mcool'),
    dict(id='larval_brain', title='Larval brain Micro-C', path=MCOOL_DIR / 'larval_brain_merge.mcool'),
]
MICROC_MIN, MICROC_MAX = -8, -3  # natural-log scale, same across maps
# Optional fixed (min, max) signal limits per group, shared across panels.
# None computes the full visible range, shared by WT/RNAi within a panel.
CHIP_LIMITS = {key: None for key in ['embryo_sry', 'embryo_beaf', 'lb_sry', 'lb_beaf']}
WIDTH_CM, FONT_SIZE, DPI = 40, 11, 300


In [ ]:
def environment():
    env = os.environ.copy()
    env['MPLBACKEND'] = 'Agg'
    env['MPLCONFIGDIR'] = str(OUTPUT / '.mplconfig')
    return env

def resolve_bigwigs():
    tracks = []
    for track in CHIP:
        matches = sorted(BASE.glob(track['prefix'] + '*.bigwig'))
        if len(matches) != 1:
            raise ValueError(f"Expected one bigWig for {track['prefix']}; found {matches}")
        tracks.append(dict(track, path=matches[0]))
    for path in [GTF, BOUNDARIES, PYGENOMETRACKS, PYTHON] + [m['path'] for m in MICROC]:
        if not path.is_file():
            raise FileNotFoundError(path)
    return tracks

def parse_region(region):
    chrom, span = region.split(':')
    start, end = map(int, span.split('-'))
    if not 0 <= start < end:
        raise ValueError(region)
    return chrom, start, end

def calculate_depth(region):
    # Standard formula from the original plotting notebook.
    _, start, end = parse_region(region)
    return int((end - start) / 9000) * 7000

# Read bigWig ranges in the existing plotting environment, without requiring
# pyBigWig in the notebook kernel. This helper only runs when called below.
def signal_ranges(tracks, region):
    import json
    chrom, start, end = parse_region(region)
    helper = """
import json, sys, pyBigWig, numpy as np
request=json.loads(sys.argv[1]); result={}
for t in request['tracks']:
    with pyBigWig.open(t['path']) as bw:
        chrom=request['chrom']; names=bw.chroms()
        if chrom not in names:
            chrom=chrom[3:] if chrom.startswith('chr') else 'chr'+chrom
        if chrom not in names or request['end'] > names[chrom]:
            raise ValueError('Region not available in '+t['path'])
        values=np.asarray(bw.values(chrom,request['start'],request['end']),dtype=float)
        values=values[np.isfinite(values)]
        if not values.size:
            raise ValueError('No finite signal in '+t['path'])
        result[t['id']]=[float(values.min()),float(values.max())]
print(json.dumps(result))
"""
    request=dict(chrom=chrom,start=start,end=end,tracks=[dict(id=t['id'],path=str(t['path'])) for t in tracks])
    result=subprocess.run([str(PYTHON), '-c', helper, json.dumps(request)],env=environment(),check=True,capture_output=True,text=True)
    ranges=json.loads(result.stdout)
    limits={}
    for group in CHIP_LIMITS:
        if CHIP_LIMITS[group] is not None:
            limits[group]=CHIP_LIMITS[group]
        else:
            members=[ranges[t['id']] for t in tracks if t['group']==group]
            low=min(0, min(v[0] for v in members))
            high=max(0, max(v[1] for v in members))
            limits[group]=(low * 1.05, high * 1.05 if high > 0 else 1)
    return limits

def track_file_link(source):
    # This pyGenomeTracks version splits INI file values on whitespace.
    # Use stable, space-free symlinks; never rename or copy the source data.
    import hashlib
    source = Path(source).resolve(strict=True)
    folder = OUTPUT / 'track_files'
    folder.mkdir(parents=True, exist_ok=True)
    name = hashlib.sha256(str(source).encode()).hexdigest()[:20] + source.suffix
    link = folder / name
    if any(char.isspace() for char in str(link)):
        raise ValueError('OUTPUT must be a path without whitespace for this pyGenomeTracks version')
    if link.is_symlink():
        if link.resolve() != source:
            raise ValueError(f'Unexpected symlink target: {link}')
    elif link.exists():
        raise FileExistsError(link)
    else:
        link.symlink_to(source)
    return link

def make_config(tracks, region, limits, resolution, microc_depth=None, microc_height=7):
    depth = calculate_depth(region) if microc_depth is None else microc_depth
    if not isinstance(depth, int) or isinstance(depth, bool) or depth <= 0:
        raise ValueError('Depth must be positive. For regions shorter than 9000 bp, set microc_depth explicitly in the panel cell.')
    if not isinstance(microc_height, (int, float)) or not math.isfinite(microc_height) or microc_height <= 0:
        raise ValueError('microc_height must be a positive height in cm')
    sections=[]
    for t in tracks:
        low, high=limits[t['group']]
        sections.append(f"""[{t['id']}]
file = {track_file_link(t['path'])}
file_type = bigwig
title = {t['title']}
height = 1.3
color = darkgreen
min_value = {low}
max_value = {high}
number_of_bins = 2000
summary_method = mean
show_data_range = true
""")
    sections.append(f"""[genes]
file = {track_file_link(GTF)}
file_type = gtf
title = FlyBase r6.59
height = 3
prefered_name = gene_symbol
color = darkgreen
color_utr = gray
border_color = black
style = flybase
labels = true
fontsize = 10
merge_transcripts = true
merge_overlapping_exons = true
all_labels_inside = true
""")
    sections.append(f"""[boundaries]
file = {track_file_link(BOUNDARIES)}
file_type = bed
title = Embryo cluster 2 boundaries
height = 0.6
color = black
border_color = black
display = collapsed
labels = false
""")
    for m in MICROC:
        sections.append(f"""[{m['id']}_microc]
file = {track_file_link(m['path'])}::/resolutions/{resolution}
file_type = hic_matrix
title = {m['title']}
height = {microc_height}
colormap = YlOrRd
transform = log
min_value = {MICROC_MIN}
max_value = {MICROC_MAX}
depth = {depth}
show_masked_bins = true
rasterize = true
""")
    sections.append('[x-axis]\nwhere = bottom\nfontsize = 12\n')
    return '\n'.join(sections)

def plot_panel(panel_id, tracks, region, resolution, dpi=300, microc_depth=None, microc_height=7):
    panel=PANELS[panel_id]
    parse_region(region)
    if not isinstance(resolution, int) or resolution <= 0:
        raise ValueError('resolution must be a positive bin size in bp')
    OUTPUT.mkdir(parents=True, exist_ok=True)
    limits=signal_ranges(tracks, region)
    ini=OUTPUT / f'Figure4_{panel_id}.ini'
    ini.write_text(make_config(tracks, region, limits, resolution, microc_depth=microc_depth, microc_height=microc_height))
    for ext in ['png','pdf']:
        output=OUTPUT / f'Figure4_{panel_id}.{ext}'
        cmd=[str(PYGENOMETRACKS),'--tracks',str(ini),'--region',region,
             '--outFileName',str(output),'--width',str(WIDTH_CM),
             '--fontSize',str(FONT_SIZE),'--dpi',str(dpi),'--trackLabelFraction','0.22']
        result=subprocess.run(cmd,env=environment(),capture_output=True,text=True)
        (OUTPUT / f'Figure4_{panel_id}_{ext}.log').write_text(result.stdout + result.stderr)
        if result.returncode:
            detail = (result.stderr or result.stdout)[-6000:]
            raise RuntimeError(f'pyGenomeTracks failed for panel {panel_id} ({ext}):\n{detail}')
    print(panel_id, panel['label'], region, f'{resolution} bp bins', limits)
    return OUTPUT / f'Figure4_{panel_id}.png'


## Panel a — Alh 

In [ ]:
from IPython.display import Image, display

# Settings for panel a only.
region = 'chr3R:7084390-7134390'
resolution = 200  # Micro-C bin size in bp; must exist in both .mcool files
dpi = 300         # Export image resolution

microc_depth = None  # Automatic depth, independent of earlier panel execution.
microc_height = 9    # Display height in cm for each Micro-C map

tracks = resolve_bigwigs()
display(Image(filename=str(plot_panel('a', tracks, region=region,
                                     resolution=resolution, dpi=dpi,
                                     microc_depth=microc_depth,
                                     microc_height=microc_height))))


## Panel b Tdrd3


In [ ]:
from IPython.display import Image, display

# Settings for panel b only.
region = 'chr3L:14763228-14813228'
resolution = 200  # Micro-C bin size in bp; must exist in both .mcool files
dpi = 300         # Export image resolution

microc_depth = None  # Automatic depth, independent of earlier panel execution.
microc_height = 9    # Display height in cm for each Micro-C map

tracks = resolve_bigwigs()
display(Image(filename=str(plot_panel('b', tracks, region=region,
                                     resolution=resolution, dpi=dpi,
                                     microc_depth=microc_depth,
                                     microc_height=microc_height))))


## Panel c — RpS2 


In [ ]:
from IPython.display import Image, display

# Settings for panel c only.
region = 'chr2L:9871908-9921908'
resolution = 100  # Micro-C bin size in bp; must exist in both .mcool files
dpi = 300         # Export image resolution
microc_depth = None
microc_height = 9    # Display height in cm for each Micro-C map
tracks = resolve_bigwigs()
display(Image(filename=str(plot_panel('c', tracks, region=region,
                                     resolution=resolution, dpi=dpi,
                                     microc_depth=microc_depth,
                                     microc_height=microc_height))))


## Panel d — RpS12 


In [ ]:
from IPython.display import Image, display

# Settings for panel d only.
region = 'chr3L:12999057-13049057'
resolution = 200  # Micro-C bin size in bp; must exist in both .mcool files
dpi = 300         # Export image resolution
microc_depth = None  # Maximum contact distance in bp, e.g. 20000; None = automatic
microc_height = 9    # Display height in cm for each Micro-C map

tracks = resolve_bigwigs()
display(Image(filename=str(plot_panel('d', tracks, region=region,
                                     resolution=resolution, dpi=dpi,
                                     microc_depth=microc_depth,
                                     microc_height=microc_height))))
